In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from tqdm import tqdm
from scipy.stats import pearsonr
from collections import defaultdict
import gseapy as gp
import anndata
import pandas as pd
from scipy import stats
from evaluation.eval_utils import *
# from plot_utils import *
from adjustText import adjust_text
from matplotlib.colors import LinearSegmentedColormap
#from plottable import ColumnDefinition, Table
#from plottable.cmap import normed_cmap
#from plottable.formatters import decimal_to_percent
#from plottable.plots import bar, percentile_bars, percentile_stars, progress_donut
import matplotlib
import os

sns.set_style('whitegrid')

In [ ]:
# Load dataset and analyze control cell subsampling effect on correlation


np.random.seed(42)

# Use the same dataset and setup
dataset_name = 'replogle_rpe1_v2_2022'
dataset = 'ReplogleRPE1_v2'
seed = 1
file = f'../data/{dataset_name}/{dataset_name}_{seed}.h5ad'
adata = anndata.read_h5ad(file)
outdir = '../results'
method = "nonctl-mean"

train_adata = adata[adata.obs['split'] == 'train']
control_adata = train_adata[train_adata.obs['control'] == 1]
pert_adata = train_adata[train_adata.obs['control'] == 0]

post_gt_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-gt.csv', index_col=[0, 1])
post_pred_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv', index_col=[0, 1])

#print(post_gt_df.head())

conditions = post_gt_df.index.get_level_values('condition').unique()
condition = conditions[0]

# Get control cells and perturbed cells
train_adata = adata[adata.obs['split'] == 'train']
control_adata = train_adata[train_adata.obs['control'] == 1]
#pert_adata = train_adata[train_adata.obs['control'] == 0]

X_true = post_gt_df.loc[condition].values[0]
X_pred = post_pred_df.loc[condition].values[0]

# Get one perturbation (condition index 7)
pert_cells_specific = pert_adata[pert_adata.obs['condition'] == condition]

# Define n_0 values from 50 to 430 every 10
n_control_values = np.arange(50, 2500, 100)
#n_control_values = np.arange(50, 430, 10)

# Store average correlations
avg_correlations = []
std_correlations = []
avg_correlations2 = []
std_correlations2 = []
avg_correlations3 = []
std_correlations3 = []

# For each n_0 value
for n_control in n_control_values:
    correlations = []
    correlations2 = []
    correlations3 = []
    
    # Run 25 repetitions
    for rep in range(25):
        # Subsample control cells to n_control
        if n_control < len(control_adata):
            indices = np.random.choice(control_adata.n_obs, n_control, replace=False)
            control_subset = control_adata[indices]
        else:
            control_subset = control_adata
        
        # Compute control mean from subsampled cells
        control_mean = np.array(control_subset.X.mean(axis=0))[0]
        
        # Compute predicted shift (mean of all perturbations) - control mean
        predicted_shift = X_pred - control_mean

        # Now split the subsampled control cells into two groups and compute predicted shift using one group, and true shift using the other group
        n_control_total = len(control_subset)
        n_group1 = n_control_total // 2
        n_group2 = n_control_total - n_group1
        control_indices = np.arange(n_control_total)
        np.random.shuffle(control_indices)
        group1_indices = control_indices[:n_group1]
        group2_indices = control_indices[n_group1:]
        control_mean1 = np.array(control_subset[group1_indices].X.mean(axis=0))[0]
        control_mean2 = np.array(control_subset[group2_indices].X.mean(axis=0))[0]
        pred_shift2 = X_pred - control_mean1
        true_shift2 = X_true - control_mean2
        corr2 = np.corrcoef(pred_shift2, true_shift2)[0, 1]
        correlations2.append(corr2)
        
        # Compute true shift (mean of specific perturbation) - control mean
        true_shift = X_true - control_mean
        
        # Compute correlation
        corr = np.corrcoef(predicted_shift, true_shift)[0, 1]
        correlations.append(corr)

        # Numerator using split control means, denominator using original control mean
        num = np.sum((pred_shift2 - pred_shift2.mean()) * (true_shift2 - true_shift2.mean()))
        denom = np.sqrt(np.sum((predicted_shift - predicted_shift.mean())**2) * np.sum((true_shift - true_shift.mean())**2))
        #num of corr using control_mean1 and control_mean2
        num = np.sum((pred_shift2 - pred_shift2.mean()) * (true_shift2 - true_shift2.mean()))

        corr3 = num / denom
        correlations3.append(corr3)

    avg_corr = np.mean(correlations)
    std_corr = np.std(correlations)
    avg_correlations.append(avg_corr)
    std_correlations.append(std_corr)
    
    avg_corr2 = np.mean(correlations2)
    std_corr2 = np.std(correlations2)
    avg_correlations2.append(avg_corr2)
    std_correlations2.append(std_corr2)

    avg_corr3 = np.mean(correlations3)
    std_corr3 = np.std(correlations3)
    avg_correlations3.append(avg_corr3)
    std_correlations3.append(std_corr3)
    
    print(f"n_0 = {n_control}: r = {avg_corr:.4f} ± {std_corr:.4f}, r2 = {avg_corr2:.4f} ± {std_corr2:.4f}, r3 = {avg_corr3:.4f} ± {std_corr3:.4f}")

# Create plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(n_control_values, avg_correlations, yerr=std_correlations, 
            fmt='o-', linewidth=2.5, markersize=8, color='steelblue', 
            capsize=5, capthick=2, label='Same control mean')

ax.errorbar(n_control_values, avg_correlations2, yerr=std_correlations2, 
            fmt='s-', linewidth=2.5, markersize=8, color='red', 
            capsize=5, capthick=2, label='Split control mean')

ax.errorbar(n_control_values, avg_correlations3, yerr=std_correlations3, 
            fmt='^-', linewidth=2.5, markersize=8, color='green', 
            capsize=5, capthick=2, label='Mixed (num split, denom original)')

ax.set_xlabel('Number of control cells ($n_0$)', fontsize=13, fontweight='bold')
ax.set_ylabel('Pearson correlation', fontsize=13, fontweight='bold')
ax.set_title(f'Effect of Control Cell Subsampling on Correlation\n({condition})', 
             fontsize=14, fontweight='bold')

ax.legend(fontsize=12, frameon=True, edgecolor='black')
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax.set_axisbelow(True)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

plt.tight_layout()
#Save as .csv in ../results
results_df = pd.DataFrame({
    'n_control': n_control_values,
    'avg_corr_same': avg_correlations,
    'std_corr_same': std_correlations,
    'avg_corr_split': avg_correlations2,
    'std_corr_split': std_correlations2,
    'avg_corr_mixed': avg_correlations3,
    'std_corr_mixed': std_correlations3
})

results_df.to_csv('../results/replogle_subsample_example.csv', index=False)

plt.savefig('control_subsampling_correlation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Load dataset and analyze control cell subsampling effect on correlation


np.random.seed(42)

# Use the same dataset and setup
dataset_name = 'tian_CRISPRi_2021_scperturb'
dataset = 'TianCRISPRi2021'
seed = 1
file = f'../data/{dataset_name}/{dataset_name}_{seed}.h5ad'
adata = anndata.read_h5ad(file)
outdir = '../results'
method = "nonctl-mean"

train_adata = adata[adata.obs['split'] == 'train']
control_adata = train_adata[train_adata.obs['control'] == 1]
pert_adata = train_adata[train_adata.obs['control'] == 0]

post_gt_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-gt.csv', index_col=[0, 1])
post_pred_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv', index_col=[0, 1])

#print(post_gt_df.head())

conditions = post_gt_df.index.get_level_values('condition').unique()
condition = conditions[7]

# Get control cells and perturbed cells
train_adata = adata[adata.obs['split'] == 'train']
control_adata = train_adata[train_adata.obs['control'] == 1]
#pert_adata = train_adata[train_adata.obs['control'] == 0]

X_true = post_gt_df.loc[condition].values[0]
X_pred = post_pred_df.loc[condition].values[0]

# Get one perturbation (condition index 7)
pert_cells_specific = pert_adata[pert_adata.obs['condition'] == condition]

# Define n_0 values from 50 to 430 every 10
#n_control_values = np.arange(50, 2500, 100)
n_control_values = np.arange(50, 430, 10)

# Store average correlations
avg_correlations = []
std_correlations = []
avg_correlations2 = []
std_correlations2 = []
avg_correlations3 = []
std_correlations3 = []

# For each n_0 value
for n_control in n_control_values:
    correlations = []
    correlations2 = []
    correlations3 = []
    
    # Run 25 repetitions
    for rep in range(25):
        # Subsample control cells to n_control
        if n_control < len(control_adata):
            indices = np.random.choice(control_adata.n_obs, n_control, replace=False)
            control_subset = control_adata[indices]
        else:
            control_subset = control_adata
        
        # Compute control mean from subsampled cells
        control_mean = np.array(control_subset.X.mean(axis=0))[0]
        
        # Compute predicted shift (mean of all perturbations) - control mean
        predicted_shift = X_pred - control_mean

        # Now split the subsampled control cells into two groups and compute predicted shift using one group, and true shift using the other group
        n_control_total = len(control_subset)
        n_group1 = n_control_total // 2
        n_group2 = n_control_total - n_group1
        control_indices = np.arange(n_control_total)
        np.random.shuffle(control_indices)
        group1_indices = control_indices[:n_group1]
        group2_indices = control_indices[n_group1:]
        control_mean1 = np.array(control_subset[group1_indices].X.mean(axis=0))[0]
        control_mean2 = np.array(control_subset[group2_indices].X.mean(axis=0))[0]
        pred_shift2 = X_pred - control_mean1
        true_shift2 = X_true - control_mean2
        corr2 = np.corrcoef(pred_shift2, true_shift2)[0, 1]
        correlations2.append(corr2)
        
        # Compute true shift (mean of specific perturbation) - control mean
        true_shift = X_true - control_mean
        
        # Compute correlation
        corr = np.corrcoef(predicted_shift, true_shift)[0, 1]
        correlations.append(corr)

        # Numerator using split control means, denominator using original control mean
        num = np.sum((pred_shift2 - pred_shift2.mean()) * (true_shift2 - true_shift2.mean()))
        denom = np.sqrt(np.sum((predicted_shift - predicted_shift.mean())**2) * np.sum((true_shift - true_shift.mean())**2))
        #num of corr using control_mean1 and control_mean2
        num = np.sum((pred_shift2 - pred_shift2.mean()) * (true_shift2 - true_shift2.mean()))

        corr3 = num / denom
        correlations3.append(corr3)

    avg_corr = np.mean(correlations)
    std_corr = np.std(correlations)
    avg_correlations.append(avg_corr)
    std_correlations.append(std_corr)
    
    avg_corr2 = np.mean(correlations2)
    std_corr2 = np.std(correlations2)
    avg_correlations2.append(avg_corr2)
    std_correlations2.append(std_corr2)

    avg_corr3 = np.mean(correlations3)
    std_corr3 = np.std(correlations3)
    avg_correlations3.append(avg_corr3)
    std_correlations3.append(std_corr3)
    
    print(f"n_0 = {n_control}: r = {avg_corr:.4f} ± {std_corr:.4f}, r2 = {avg_corr2:.4f} ± {std_corr2:.4f}, r3 = {avg_corr3:.4f} ± {std_corr3:.4f}")

# Create plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(n_control_values, avg_correlations, yerr=std_correlations, 
            fmt='o-', linewidth=2.5, markersize=8, color='steelblue', 
            capsize=5, capthick=2, label='Same control mean')

ax.errorbar(n_control_values, avg_correlations2, yerr=std_correlations2, 
            fmt='s-', linewidth=2.5, markersize=8, color='red', 
            capsize=5, capthick=2, label='Split control mean')

ax.errorbar(n_control_values, avg_correlations3, yerr=std_correlations3, 
            fmt='^-', linewidth=2.5, markersize=8, color='green', 
            capsize=5, capthick=2, label='Mixed (num split, denom original)')

ax.set_xlabel('Number of control cells ($n_0$)', fontsize=13, fontweight='bold')
ax.set_ylabel('Pearson correlation', fontsize=13, fontweight='bold')
ax.set_title(f'Effect of Control Cell Subsampling on Correlation\n({condition})', 
             fontsize=14, fontweight='bold')

ax.legend(fontsize=12, frameon=True, edgecolor='black')
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax.set_axisbelow(True)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

#Save as .csv in ../results 


plt.tight_layout()
results_df = pd.DataFrame({
    'n_control': n_control_values,
    'avg_corr_same': avg_correlations,
    'std_corr_same': std_correlations,
    'avg_corr_split': avg_correlations2,
    'std_corr_split': std_correlations2,
    'avg_corr_mixed': avg_correlations3,
    'std_corr_mixed': std_correlations3
})

results_df.to_csv('../results/tiancrispri_subsample_example.csv', index=False)
#plt.savefig('control_subsampling_correlation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Load dataset and analyze control cell subsampling effect on correlation


np.random.seed(42)

# Use the same dataset and setup
dataset_name = 'xu_kinetics_2024'
dataset = 'XuKinetics2024'
seed = 1
file = f'../data/{dataset_name}/{dataset_name}_{seed}.h5ad'
adata = anndata.read_h5ad(file)
outdir = '../results'
method = "nonctl-mean"

train_adata = adata[adata.obs['split'] == 'train']
control_adata = train_adata[train_adata.obs['control'] == 1]
pert_adata = train_adata[train_adata.obs['control'] == 0]

post_gt_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-gt.csv', index_col=[0, 1])
post_pred_df = pd.read_csv(f'{outdir}/{dataset}_{seed}_{method}_post-pred.csv', index_col=[0, 1])

#print(post_gt_df.head())

conditions = post_gt_df.index.get_level_values('condition').unique()
#Print all 204 conditions 
#print("RELA+ctrl" in conditions)
#print(conditions)
condition = conditions[1]
print(condition)

# Get control cells and perturbed cells
train_adata = adata[adata.obs['split'] == 'train']
control_adata = train_adata[train_adata.obs['control'] == 1]
#pert_adata = train_adata[train_adata.obs['control'] == 0]

X_true = post_gt_df.loc[condition].values[0]
X_pred = post_pred_df.loc[condition].values[0]

# Get one perturbation (condition index 7)
pert_cells_specific = pert_adata[pert_adata.obs['condition'] == condition]

# Define n_0 values from 50 to 430 every 10
n_control_values = np.arange(50, 2500, 100)
#n_control_values = np.arange(50, 430, 10)

# Store average correlations
avg_correlations = []
std_correlations = []
avg_correlations2 = []
std_correlations2 = []
avg_correlations3 = []
std_correlations3 = []

# For each n_0 value
for n_control in n_control_values:
    correlations = []
    correlations2 = []
    correlations3 = []
    
    # Run 25 repetitions
    for rep in range(25):
        # Subsample control cells to n_control
        if n_control < len(control_adata):
            indices = np.random.choice(control_adata.n_obs, n_control, replace=False)
            control_subset = control_adata[indices]
        else:
            control_subset = control_adata
        
        # Compute control mean from subsampled cells
        control_mean = np.array(control_subset.X.mean(axis=0))[0]
        
        # Compute predicted shift (mean of all perturbations) - control mean
        predicted_shift = X_pred - control_mean

        # Now split the subsampled control cells into two groups and compute predicted shift using one group, and true shift using the other group
        n_control_total = len(control_subset)
        n_group1 = n_control_total // 2
        n_group2 = n_control_total - n_group1
        control_indices = np.arange(n_control_total)
        np.random.shuffle(control_indices)
        group1_indices = control_indices[:n_group1]
        group2_indices = control_indices[n_group1:]
        control_mean1 = np.array(control_subset[group1_indices].X.mean(axis=0))[0]
        control_mean2 = np.array(control_subset[group2_indices].X.mean(axis=0))[0]
        pred_shift2 = X_pred - control_mean1
        true_shift2 = X_true - control_mean2
        corr2 = np.corrcoef(pred_shift2, true_shift2)[0, 1]
        correlations2.append(corr2)
        
        # Compute true shift (mean of specific perturbation) - control mean
        true_shift = X_true - control_mean
        
        # Compute correlation
        corr = np.corrcoef(predicted_shift, true_shift)[0, 1]
        correlations.append(corr)

        # Numerator using split control means, denominator using original control mean
        num = np.sum((pred_shift2 - pred_shift2.mean()) * (true_shift2 - true_shift2.mean()))
        denom = np.sqrt(np.sum((predicted_shift - predicted_shift.mean())**2) * np.sum((true_shift - true_shift.mean())**2))
        #num of corr using control_mean1 and control_mean2
        num = np.sum((pred_shift2 - pred_shift2.mean()) * (true_shift2 - true_shift2.mean()))

        corr3 = num / denom
        correlations3.append(corr3)

    avg_corr = np.mean(correlations)
    std_corr = np.std(correlations)
    avg_correlations.append(avg_corr)
    std_correlations.append(std_corr)
    
    avg_corr2 = np.mean(correlations2)
    std_corr2 = np.std(correlations2)
    avg_correlations2.append(avg_corr2)
    std_correlations2.append(std_corr2)

    avg_corr3 = np.mean(correlations3)
    std_corr3 = np.std(correlations3)
    avg_correlations3.append(avg_corr3)
    std_correlations3.append(std_corr3)
    
    print(f"n_0 = {n_control}: r = {avg_corr:.4f} ± {std_corr:.4f}, r2 = {avg_corr2:.4f} ± {std_corr2:.4f}, r3 = {avg_corr3:.4f} ± {std_corr3:.4f}")

# Create plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(n_control_values, avg_correlations, yerr=std_correlations, 
            fmt='o-', linewidth=2.5, markersize=8, color='steelblue', 
            capsize=5, capthick=2, label='Same control mean')

ax.errorbar(n_control_values, avg_correlations2, yerr=std_correlations2, 
            fmt='s-', linewidth=2.5, markersize=8, color='red', 
            capsize=5, capthick=2, label='Split control mean')

ax.errorbar(n_control_values, avg_correlations3, yerr=std_correlations3, 
            fmt='^-', linewidth=2.5, markersize=8, color='green', 
            capsize=5, capthick=2, label='Mixed (num split, denom original)')

ax.set_xlabel('Number of control cells ($n_0$)', fontsize=13, fontweight='bold')
ax.set_ylabel('Pearson correlation', fontsize=13, fontweight='bold')
ax.set_title(f'Effect of Control Cell Subsampling on Correlation\n({condition})', 
             fontsize=14, fontweight='bold')

ax.legend(fontsize=12, frameon=True, edgecolor='black')
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax.set_axisbelow(True)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

#Save as .csv in ../results 


plt.tight_layout()
results_df = pd.DataFrame({
    'n_control': n_control_values,
    'avg_corr_same': avg_correlations,
    'std_corr_same': std_correlations,
    'avg_corr_split': avg_correlations2,
    'std_corr_split': std_correlations2,
    'avg_corr_mixed': avg_correlations3,
    'std_corr_mixed': std_correlations3
})

results_df.to_csv('../results/xu_subsample_example.csv', index=False)
#plt.savefig('control_subsampling_correlation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()